# Atelier Preparation de Donnees Images

Objectif : construire un jeu de donnees images propre et homogene pour un modele de classification de dechets (cardboard, glass, metal, paper, plastic, trash).

## Partie 1 - Exploration du dataset

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

RAW_DIR = Path("../data/raw")
CLEANED_DIR = Path("../data/cleaned")
REPORTS_DIR = Path("../reports")

CLASSES = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
CLASSES

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

In [2]:
for classe in CLASSES:
    nb_fichiers = len(list((RAW_DIR / classe).iterdir()))
    print(f"{classe}: {nb_fichiers} fichiers")

cardboard: 169 fichiers
glass: 188 fichiers
metal: 149 fichiers
paper: 252 fichiers
plastic: 224 fichiers
trash: 50 fichiers


In [3]:
def get_format_mode_dimensions(img):
    """Retourne (format, mode, largeur, hauteur) a partir d'une image PIL deja ouverte."""
    largeur, hauteur = img.size
    return img.format, img.mode, largeur, hauteur

In [4]:
def get_pixel_std(img):
    """Retourne l'ecart-type des valeurs de pixels d'une image PIL deja ouverte."""
    array = np.array(img)
    return float(array.std())

In [5]:
MODE_TO_CHANNELS = {
    "1": 1, "L": 1, "P": 1,
    "LA": 2,
    "RGB": 3,
    "RGBA": 4, "CMYK": 4,
}

def get_nb_channels(mode):
    """Deduit le nombre de canaux a partir du mode PIL de l'image."""
    return MODE_TO_CHANNELS.get(mode, len(mode))

def get_file_size(chemin):
    """Retourne la taille du fichier en octets."""
    return os.path.getsize(chemin)

In [6]:
def extraire_metadonnees(chemin, classe):
    """Ouvre une image et retourne un dictionnaire avec ses metadonnees."""
    with Image.open(chemin) as img:
        img.load()
        format_img, mode, largeur, hauteur = get_format_mode_dimensions(img)
        ecart_type = get_pixel_std(img)
        nb_canaux = get_nb_channels(mode)

    return {
        "nom": chemin.name,
        "classe": classe,
        "format": format_img,
        "mode": mode,
        "largeur": largeur,
        "hauteur": hauteur,
        "ecart_type_pixels": ecart_type,
        "nb_canaux": nb_canaux,
        "taille_octets": get_file_size(chemin),
    }

# test sur une seule image
premiere_image = next((RAW_DIR / CLASSES[0]).iterdir())
extraire_metadonnees(premiere_image, CLASSES[0])

{'nom': 'cardboard1.jpg',
 'classe': 'cardboard',
 'format': 'JPEG',
 'mode': 'RGB',
 'largeur': 512,
 'hauteur': 384,
 'ecart_type_pixels': 40.58852860499886,
 'nb_canaux': 3,
 'taille_octets': 17333}